## Trabajo Práctico 1 - Telco Customer Churn
- **Tecnología Digital VI "Inteligencia Artificial"**
- **Integrantes:** *Arslanian Juana, Cerdeira Matías, Moreira Lourdes*  
- **Semestre:** 2026 - 2.º semestre

### Introducción y objetivos
En este trabajo abordamos el ciclo de vida completo de un problema de **Machine Learning**, priorizando el rigor metodológico y la justificación crítica en cada etapa. Nuestro objetivo es estructurar un flujo de trabajo reproducible centrado en la **preparación y limpieza de datos**, la **prevención estricta de *Data Leakage***, el diseño de un **esquema de validación robusto** y la selección de **métricas de evaluación alineadas con el impacto del negocio**.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("dataset Telco Customer Churn.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("Filas duplicadas:", df.duplicated().sum())
print("IDs duplicados:", df["customerID"].duplicated().sum())
print(df.isna().sum())

display(df.head())
df.info()


Filas: 7043
Columnas: 21
Filas duplicadas: 0
IDs duplicados: 0
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 1. Introducción al problema y Análisis Exploratorio

El dataset contiene información de **7.043 clientes** de una empresa de telecomunicaciones. Cada fila representa un cliente y las columnas describen características demográficas, servicios contratados, tipo de contrato, antigüedad, etc. de los mismos. 

El objetivo del trabajo es predecir que tan probable es que los clientes se den de baja de nuestro servicio o no (Churn). Esto es justamente un problema de clasificación binaria supervisada.

Desde el punto de vista empresarial, identificar anticipadamente a los clientes con mayor riesgo de abandono permitiría tomar medidas al respecto con ellos para poder retenerlos. 

algunas columnas no fueron reconocidas como deberían haber sido identificadas, por ejemplo `TotalCharges` mide el importe total acumulado que el cliente pagó durante el tiempo que permaneció en la empresa. Por lo que debería ser una variable numerica pero es un Str y esto se puede deber a que hubo datos que no son exactamente números sino espacios en blanco

In [2]:
# Convertimos TotalCharges a una columna numérica.
# Los valores que no puedan convertirse, se reemplazan por NaN.

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

vacios = df["TotalCharges"].isna()

# Mostramos los 11 clientes con valores nulos en TotalCharges
df.loc[vacios]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


## 2. Preprocesamiento y prevención de Data Leakage

Como podemos ver, se detectaron 11 espacios en blanco que deberían contener algún número mostrando el importe total acumulado por cada cliente. 
Ahora la pregunta es: ¿Por qué esos 11 clientes tienen `TotalCharges` en blanco?

- Principalmente se puede deber a que tienen `Tenure` (meses que el cliente lleva utilizando el servicio) en cero. Esto indica que son clientes nuevos que todavía no completaron un período de facturación y, por lo tanto, aún no acumularon cargos totales.

- Para resolver el problema decidimos convertir la columna `TotalCharges` a formato numérico e imputarles un 0 a los valores faltantes

Si nosotros no hubiesemos detectado este problema algunos modelos no podrían utilizar la columna por tener formato texto. Además, no podríamos calcular correctamente estadísticas como la media o la mediana, lo que afectaría el análisis y el entrenamiento del modelo.


In [8]:
numeric_summary = df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().T
categorical_summary = pd.DataFrame(
    {
        "categorías": df.select_dtypes(exclude="number").nunique(),
        "moda": df.select_dtypes(exclude="number").mode().iloc[0],
        "frecuencia_moda": [
            df[column].value_counts().iloc[0]
            for column in df.select_dtypes(exclude="number").columns
        ],
    }
)

print("Resumen de variables numéricas")
display(numeric_summary.round(2))
print("Resumen de variables categóricas")
display(categorical_summary)

Resumen de variables numéricas


,count,mean,std,min,25%,50%,75%,max
tenure,7043.0,32.37,24.56,0.00,9.00,29.00,55.00,72.00
MonthlyCharges,7043.0,64.76,30.09,18.25,35.50,70.35,89.85,118.75
TotalCharges,7032.0,2283.30,2266.77,18.80,401.45,1397.48,3794.74,8684.80


Resumen de variables categóricas


,categorías,moda,frecuencia_moda
customerID,7043,0002-ORFBO,1
gender,2,Male,3555
Partner,2,No,3641
Dependents,2,No,4933
PhoneService,2,Yes,6361
MultipleLines,3,No,3390
InternetService,3,Fiber optic,3096
OnlineSecurity,3,No,3498
OnlineBackup,3,No,3088
DeviceProtection,3,No,3095


Uno de los momentos donde más fácil puede ocurrir el Data Leakage es cuando manipulamos las variables de entrada. Básicamente, el **Data Leakage** pasa cuando, sin querer, usamos información del test en el entrenamiento (directa o indirectamente). Por eso el orden en el que hacemos el split y el preprocesamiento importa un montón.

Lo primero que podemos aplicar son correcciones determinísticas, o sea que no dependan de los datos en sí: convertir una variable de texto a número, corregir un valor puntual siguiendo una regla fija, etc. Recién después de eso separamos en `train` y `test`.

De ahí en adelante, cualquier transformación que necesite "aprender" algo del dataset como una imputación, un escalado, codificar variables categóricas, tiene que ajustarse solo con los datos de train. O sea, hacemos el fit sobre X_train, y con esos mismos parámetros transformamos después validación y test.

Si en cambio calculamos la media, la mediana o el desvío estándar usando todo el dataset antes de splitear, ya estamos metiendo información del test. Aunque sea indirecto, el modelo termina recibiendo la información que en teoría no debería conocer. Eso es Data Leakage, y hace que las métricas te den más optimistas de lo que serían en la realidad con datos nuevos.

Usar un Pipeline también ayuda un montón acá, sobre todo en cross-validation, porque hace que el preprocesamiento se reajuste solo con los datos de train que le tocan a cada fold.